In [1]:
# Lab2_aurora_bay_rag.py
# Jupyter Notebook (GCP) — RAG chatbot using BigQuery vector search + Gemini
#
# Pipeline:
#   GCS CSV  →  BigQuery table  →  generate embeddings  →  store in BQ
#   User question  →  embed  →  VECTOR_SEARCH in BQ  →  Gemini answer
#
# =============================================================================
# SETUP NOTES (manual steps required before running):
#
# 1. Enable APIs:
#    gcloud services enable aiplatform.googleapis.com
#    gcloud services enable bigquery.googleapis.com
#
# 2. Grant your account / service account these roles:
#    roles/bigquery.dataEditor
#    roles/bigquery.jobUser
#    roles/aiplatform.user
#
# 3. The source file is public — no GCS permissions needed:
#    gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv
# =============================================================================


In [2]:
# %% [markdown]
# ## Lab2 — Aurora Bay FAQ RAG Chatbot

In [3]:
# Install dependencies
# Run once; restart kernel after.
# !pip install --quiet google-cloud-bigquery google-cloud-aiplatform


In [1]:
# Imports
import time
import vertexai
from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
from vertexai.generative_models import GenerativeModel, GenerationConfig

In [5]:
# Configuration
# *** Update PROJECT_ID for your environment ***
PROJECT_ID     = "qwiklabs-gcp-00-16d0362ac1ac"
LOCATION       = "us-east4"

BQ_DATASET     = "aurora_bay"
BQ_TABLE       = "AuroraBayFAQ"
BQ_TABLE_REF   = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"

GCS_SOURCE     = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

EMBED_MODEL    = "text-embedding-004"
GEMINI_MODEL   = "gemini-2.5-flash"
EMBED_DIM      = 768          # text-embedding-004 output dimension
TOP_K          = 5            # number of FAQ results to pass to Gemini

print("Configuration loaded.")
print(f"  Source   : {GCS_SOURCE}")
print(f"  BQ table : {BQ_TABLE_REF}")
print(f"  Embedding: {EMBED_MODEL}  dim={EMBED_DIM}")
print(f"  Gemini   : {GEMINI_MODEL}  top_k={TOP_K}")

Configuration loaded.
  Source   : gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv
  BQ table : qwiklabs-gcp-00-16d0362ac1ac.aurora_bay.AuroraBayFAQ
  Embedding: text-embedding-004  dim=768
  Gemini   : gemini-2.5-flash  top_k=5


In [6]:
# Initialize clients
vertexai.init(project=PROJECT_ID, location=LOCATION)
bq_client    = bigquery.Client(project=PROJECT_ID)
embed_model  = TextEmbeddingModel.from_pretrained(EMBED_MODEL)
gemini_model = GenerativeModel(GEMINI_MODEL)
print("Clients initialized.")

Clients initialized.


/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [7]:
# Create BigQuery dataset and load CSV from GCS
# Creates the dataset if it doesn't exist, then loads the CSV.
# autodetect=True infers the schema from the first rows of the file.

dataset_ref = bigquery.DatasetReference(PROJECT_ID, BQ_DATASET)
try:
    bq_client.get_dataset(dataset_ref)
    print(f"Dataset already exists: {BQ_DATASET}")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"
    bq_client.create_dataset(dataset)
    print(f"Dataset created: {BQ_DATASET}")

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,     # header row
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # replace on re-run
)

print(f"Loading {GCS_SOURCE} → {BQ_TABLE_REF} ...")
load_job = bq_client.load_table_from_uri(GCS_SOURCE, BQ_TABLE_REF, job_config=job_config)
load_job.result()   # blocks until complete

table = bq_client.get_table(BQ_TABLE_REF)
print(f"Loaded {table.num_rows} rows.")
print("Schema detected:")
for field in table.schema:
    print(f"  {field.name:30s} {field.field_type}")

Dataset created: aurora_bay
Loading gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv → qwiklabs-gcp-00-16d0362ac1ac.aurora_bay.AuroraBayFAQ ...
Loaded 50 rows.
Schema detected:
  string_field_0                 STRING
  string_field_1                 STRING


In [8]:
# Preview loaded data
print("\nFirst 5 rows:")
preview = bq_client.query(f"SELECT * FROM `{BQ_TABLE_REF}` LIMIT 5").result()
for row in preview:
    print(dict(row))


First 5 rows:
{'string_field_0': 'When was Aurora Bay founded?', 'string_field_1': 'Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.'}
{'string_field_0': 'What is the population of Aurora Bay?', 'string_field_1': 'Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.'}
{'string_field_0': 'Where is the Aurora Bay Town Hall located?', 'string_field_1': 'The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.'}
{'string_field_0': 'Who is the current mayor of Aurora Bay?', 'string_field_1': 'The current mayor is Linda Greenwood, elected in 2021 for a four-year term.'}
{'string_field_0': 'What are the primary industries in Aurora Bay?', 'string_field_1': 'The primary industries include commercial fishing, tourism, and small-scale logging in the nearby forests.'}


In [9]:
# Generate embeddings for each FAQ row
# Concatenates every column into a single string for embedding so the vector
# captures both the question and answer context. Batches API calls to stay
# within the 250-text-per-request limit for text-embedding-004.

def rows_to_text(row: dict) -> str:
    """Flatten all fields into a single string for embedding."""
    return " | ".join(f"{k}: {v}" for k, v in row.items() if v is not None)


def embed_batch(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts; returns list of float vectors."""
    inputs = [TextEmbeddingInput(t, task_type="RETRIEVAL_DOCUMENT") for t in texts]
    results = embed_model.get_embeddings(inputs)
    return [r.values for r in results]


BATCH_SIZE = 100   # well within the 250-text API limit

print("Fetching all rows from BigQuery...")
all_rows = list(bq_client.query(f"SELECT * FROM `{BQ_TABLE_REF}`").result())
print(f"  {len(all_rows)} rows fetched.")

records = []   # will hold {original fields + embedding}
total = len(all_rows)

for batch_start in range(0, total, BATCH_SIZE):
    batch = all_rows[batch_start : batch_start + BATCH_SIZE]
    texts = [rows_to_text(dict(r)) for r in batch]
    vectors = embed_batch(texts)
    for row, vector in zip(batch, vectors):
        rec = dict(row)
        rec["embedding"] = vector
        records.append(rec)
    print(f"  Embedded {min(batch_start + BATCH_SIZE, total)}/{total} rows...")
    time.sleep(0.5)   # gentle rate-limit buffer

print(f"Embeddings generated for all {len(records)} rows.")

Fetching all rows from BigQuery...
  50 rows fetched.
  Embedded 50/50 rows...
Embeddings generated for all 50 rows.


In [10]:
# Write embeddings back to BigQuery
# Recreates the table with the embedding column added.
# BigQuery requires ARRAY<FLOAT64> for VECTOR_SEARCH.

existing_schema = bq_client.get_table(BQ_TABLE_REF).schema
embedding_field = bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED")
new_schema = existing_schema + [embedding_field]

job_config_embed = bigquery.LoadJobConfig(
    schema=new_schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print(f"Writing {len(records)} rows with embeddings to {BQ_TABLE_REF} ...")
load_job2 = bq_client.load_table_from_json(records, BQ_TABLE_REF, job_config=job_config_embed)
load_job2.result()

table = bq_client.get_table(BQ_TABLE_REF)
print(f"Table updated — {table.num_rows} rows, {len(table.schema)} columns.")


Writing 50 rows with embeddings to qwiklabs-gcp-00-16d0362ac1ac.aurora_bay.AuroraBayFAQ ...
Table updated — 50 rows, 3 columns.


In [12]:
# Create BigQuery vector index (skips automatically if table is too small)
# IVF requires 5 000+ rows. For smaller tables VECTOR_SEARCH does a full scan
# which is equally correct and fast enough for lab-scale data.

INDEX_NAME    = f"{BQ_DATASET}_faq_embedding_idx"
MIN_IDX_ROWS  = 5_000

row_count = bq_client.get_table(BQ_TABLE_REF).num_rows
print(f"Table has {row_count} rows (index requires {MIN_IDX_ROWS}+).")

if row_count < MIN_IDX_ROWS:
    print("Skipping vector index — VECTOR_SEARCH will use a full scan.")
else:
    print("Creating vector index...")
    try:
        bq_client.query(
            f"DROP VECTOR INDEX IF EXISTS `{INDEX_NAME}` ON `{BQ_TABLE_REF}`"
        ).result()
        bq_client.query(f"""
            CREATE VECTOR INDEX `{INDEX_NAME}`
            ON `{BQ_TABLE_REF}`(embedding)
            OPTIONS (distance_type = 'COSINE', index_type = 'IVF')
        """).result()
        print("Index created. Waiting 60s for it to become active...")
        time.sleep(60)
    except Exception as e:
        print(f"Index creation failed — falling back to full scan.\n  Reason: {e}")

print("Ready.")


Table has 50 rows (index requires 5000+).
Skipping vector index — VECTOR_SEARCH will use a full scan.
Ready.


In [13]:
# Helper: vector search + Gemini answer

def search_faqs(question: str) -> list[dict]:
    """Embed the question and run VECTOR_SEARCH against the FAQ table."""
    q_vector = embed_model.get_embeddings(
        [TextEmbeddingInput(question, task_type="RETRIEVAL_QUERY")]
    )[0].values

    # Format the vector literal for BigQuery SQL
    vector_literal = "[" + ", ".join(str(v) for v in q_vector) + "]"

    sql = f"""
        SELECT
            base.*,
            distance
        FROM
            VECTOR_SEARCH(
                TABLE `{BQ_TABLE_REF}`,
                'embedding',
                (SELECT {vector_literal} AS embedding),
                top_k => {TOP_K},
                distance_type => 'COSINE'
            )
        ORDER BY distance ASC
    """
    rows = list(bq_client.query(sql).result())
    return [dict(r) for r in rows]


def build_context(faq_rows: list[dict]) -> str:
    """Format retrieved FAQ rows into a readable context block."""
    lines = []
    for i, row in enumerate(faq_rows, 1):
        # Exclude the embedding vector from the context passed to Gemini
        fields = {k: v for k, v in row.items() if k not in ("embedding", "distance")}
        lines.append(f"[Result {i}]")
        for k, v in fields.items():
            lines.append(f"  {k}: {v}")
    return "\n".join(lines)


def ask(question: str) -> str:
    """Full RAG pipeline: search → build prompt → Gemini answer."""
    print(f"\nQuestion: {question}")
    print("Searching FAQs...")

    faq_rows = search_faqs(question)
    context  = build_context(faq_rows)

    prompt = f"""You are a helpful assistant for Aurora Bay.
Answer the user's question using ONLY the FAQ information provided below.
If the answer is not in the FAQs, say so clearly.

--- Aurora Bay FAQ Context ---
{context}
--- End of Context ---

User question: {question}
Answer:"""

    response = gemini_model.generate_content(
        prompt,
        generation_config=GenerationConfig(temperature=0.2, max_output_tokens=512),
    )
    return response.text


In [17]:
# Chatbot loop
# Edit QUESTIONS or replace with input() for an interactive version.

QUESTIONS = [
    "What are the hours of operation for the Aurora Bay Public Library?",
    "What are the hours of operation for the local fishermen's market?",
    "What is the local business tax rate in Aurora Bay?",
    "Is there an airport in Aurora Bay?",
    "What is your cancellation policy?",
]

print("=" * 70)
print("Lab2 — Aurora Bay FAQ Chatbot")
print("=" * 70)

for q in QUESTIONS:
    answer = ask(q)
    print(f"\nAnswer:\n{answer}")
    print("-" * 70)


Lab2 — Aurora Bay FAQ Chatbot

Question: What are the hours of operation for the Aurora Bay Public Library?
Searching FAQs...

Answer:
The Aurora Bay Public Library is open Monday through Friday from 9 AM to 6 PM, and on Saturdays from 10 AM to 4 PM. It’s closed on Sundays and major holidays.
----------------------------------------------------------------------

Question: What are the hours of operation for the local fishermen's market?
Searching FAQs...

Answer:
The fishermen’s market runs from 8 AM to 2 PM.
----------------------------------------------------------------------

Question: What is the local business tax rate in Aurora Bay?
Searching FAQs...

Answer:
The local business tax rate in Aurora Bay is 2% on net profits.
----------------------------------------------------------------------

Question: Is there an airport in Aurora Bay?
Searching FAQs...

Answer:
Yes, there is an airport in Aurora Bay. Most visitors arrive via regional flights into Aurora Bay Airport.
---------

In [ ]:
# Interactive mode (optional)
# Uncomment to run a free-form chat session. Type 'quit' to exit.

# while True:
#     user_input = input("\nYou: ").strip()
#     if user_input.lower() in ("quit", "exit", "q"):
#         print("Goodbye!")
#         break
#     if user_input:
#         print(f"\nAssistant: {ask(user_input)}")